# 1.0 - Khảo sát ban đầu: bằng chứng 5 điều kiện dữ liệu

**Đề tài:** Giá nhà đất Việt Nam sau sáp nhập tỉnh

**Câu hỏi phân tích:** Sau sáp nhập còn 34 tỉnh, thành, giá rao bán căn hộ trung vị (triệu đồng/m²)
chênh nhau bao nhiêu lần giữa tỉnh cao nhất và thấp nhất, và một lao động ở mỗi tỉnh cần bao nhiêu
năm thu nhập để mua một căn 70 m²?

**Mục đích notebook:** chứng minh dữ liệu đạt đủ **5 điều kiện ở mục 3.11 của đề**. Mọi con số
trong bản đăng ký đều lấy từ kết quả chạy notebook này.

**Nguyên tắc:**
- Chỉ **đọc** `data/raw/` và `data/external/`, không ghi tệp nào.
- Không nạp `name` và `description` (văn bản tự do, có thể có thông tin cá nhân).
- Chạy lại bằng *Restart & Run All*, từ thư mục `notebooks/`.

### Năm điều kiện của đề (mục 3.11)

| # | Điều kiện |
|---|---|
| 1 | Ít nhất **hai nguồn độc lập** phải được hợp nhất bằng mã. |
| 2 | Có một tệp từ **5 MB** trở lên, hoặc có một nguồn là **REST API có phân trang**. |
| 3 | Từ **20.000 dòng** trở lên sau khi hợp nhất (nếu ít hơn thì phải có từ bốn bảng trở lên cần ghép). |
| 4 | Có ít nhất **ba trong bốn khuyết tật**: giá trị thiếu, bản ghi trùng, kiểu dữ liệu sai, danh mục chưa chuẩn hoá. |
| 5 | Có ít nhất một cột **ngày hoặc giờ**, và **giấy phép** cho phép dùng cho mục đích học tập. |

### Thứ tự các phần trong notebook

| Phần | Nội dung | Điều kiện |
|---|---|---|
| 0 | Thiết lập | - |
| 1 | Kích thước tệp | **2** |
| 2 | Nạp ba nguồn | - |
| 3 | Tổng quan dữ liệu sau khi nạp | - |
| 4 | Cột ngày giờ | **5** (ngày giờ) |
| 5 | Giấy phép sử dụng | **5** (giấy phép) |
| 6 | Bốn loại khuyết tật | **4** |
| 7 | Ghép hai nguồn bằng khoá | **1** |
| 8 | Số dòng sau hợp nhất | **3** |
| 9 | Tổng kết theo thứ tự điều kiện 1 → 5 | 1-5 |

## Phần 0 - Thiết lập và làm quen dữ liệu thô

Trước khi kiểm bất kỳ điều kiện nào, notebook **tự nhìn** dữ liệu thô: mỗi nguồn có những cột / khoá
nào, kiểu gì, vài dòng mẫu trông ra sao. Chỉ sau bước này mới **chọn** cột cần dùng — để mọi lựa chọn
về sau đều có căn cứ ngay trong notebook, không dựa vào hiểu biết có sẵn từ bên ngoài.

In [ ]:
import json
import re
import time
import unicodedata
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

pd.set_option("display.max_columns", None, "display.width", 200)

GOC = Path("..")
THU_MUC_TIN_RAO = GOC / "data" / "raw" / "tin_rao"
TEP_THU_NHAP = GOC / "data" / "raw" / "thu_nhap.json"
TEP_BANG_TRA = GOC / "data" / "external" / "converter_2025.json"

for p in [THU_MUC_TIN_RAO, TEP_THU_NHAP, TEP_BANG_TRA]:
    print(p, "->", "có" if p.exists() else "KHÔNG CÓ")

..\data\raw\tin_rao -> có
..\data\raw\thu_nhap.json -> có
..\data\external\converter_2025.json -> có


### a. Tin rao

In [ ]:
# lược đồ về cột
cac_phan = sorted(THU_MUC_TIN_RAO.glob("shard_*.parquet"))
luoc_do = pq.read_schema(cac_phan[0])

bang_cot = pd.DataFrame({"cot": luoc_do.names,
                         "kieu": [str(t) for t in luoc_do.types]})
print("Số phần Parquet:", len(cac_phan))
print("Số cột:", len(bang_cot))
print("10 phần cùng lược đồ:", all(pq.read_schema(p).equals(luoc_do) for p in cac_phan))
bang_cot

Số phần Parquet: 10
Số cột: 19
10 phần cùng lược đồ: True


,cot,kieu
0,name,string
1,description,string
2,property_type_name,string
3,province_name,string
4,district_name,string
5,ward_name,string
6,street_name,string
7,project_name,string
8,price,string
9,area,double


In [ ]:
# nhìn sơ 5 dòng đầu
lo_dau = next(pq.ParquetFile(cac_phan[0]).iter_batches(batch_size=5))
with pd.option_context("display.max_colwidth", 30):
    display(lo_dau.to_pandas())

,name,description,property_type_name,province_name,district_name,ward_name,street_name,project_name,price,area,floor_count,frontage_width,house_depth,road_width,bedroom_count,bathroom_count,house_direction,balcony_direction,published_at
0,Bán nhanh nhà 6 tầng thang...,- Vị trí siêu đẹp ô tô đỗ ...,Nhà,Hà Nội,Bắc Từ Liêm,Phúc Diễn,Ngọa Long,NaN,7450000000,37.0,NaN,NaN,NaN,NaN,4.0,5.0,NaN,NaN,2025-06-01T05:12:56.941000
1,Bán gấp nhà mặt tiền 4 tần...,Nhà 4 tầng mặt tiền KDC Tạ...,Nhà,Hồ Chí Minh,8,5,Tạ Quang Bửu,NaN,9100000000,48.0,4.0,4.0,NaN,NaN,3.0,3.0,NaN,NaN,2025-06-06T04:37:25.903000
2,Bán biệt thự 300m2 gần sân...,Bán biệt thự Bình Minh diệ...,Biệt thự/Nhà liền kề,Vĩnh Long,Bình Minh,NaN,NaN,NaN,20000000000,300.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-06-15T18:58:07.455000
3,"Cần bán căn hộ CC Thái An,...",Chung cư Thái An đường Ngu...,Căn hộ chung cư,Hồ Chí Minh,12,Đông Hưng Thuận,Nguyễn Văn Quá,Thái An Apartment,1300000000,44.0,NaN,NaN,NaN,NaN,1.0,1.0,NaN,NaN,2025-06-17T05:32:15.491000
4,"MTKD p.Tân Thành, Tân Phú,...","MTKD p.Tân Thành, Tân Phú,...",Nhà,Hồ Chí Minh,Tân Phú,Tân Thành,Nguyễn Hậu,NaN,9000000000,60.0,NaN,NaN,NaN,NaN,2.0,3.0,NaN,NaN,2025-06-06T04:06:17.378000


In [ ]:
# các loại bất động sản
loai_nha = pd.read_parquet(THU_MUC_TIN_RAO, columns=["property_type_name"])
loai_nha["property_type_name"].value_counts().to_frame("so_tin")

,so_tin
property_type_name,
Nhà,1575536
Đất,832766
Căn hộ chung cư,762800
Biệt thự/Nhà liền kề,269782
Shophouse,59860


### b Thu nhập

In [ ]:
# cấu trúc tệp JSON-stat
noi_dung = TEP_THU_NHAP.read_text(encoding="utf-8")
# Tệp xuất từ PX-Web có ký tự rỗng (NUL) thừa ở cuối -> json.loads báo "Extra data"
print("3 ký tự cuối của tệp:", repr(noi_dung[-3:]), "| số ký tự NUL:", noi_dung.count("\x00"))
js = json.loads(noi_dung.rstrip("\x00"))
print("Khoá cấp ngoài:", list(js))
ds = js["dataset"]
print("Khoá trong 'dataset':", list(ds))
print("Tiêu đề bảng:", ds["label"])
for chieu in ds["dimension"]["id"]:
    nhan = list(ds["dimension"][chieu]["category"]["label"].values())
    print(f"Chiều '{chieu}': {len(nhan)} giá trị, ví dụ {nhan[:4]} ... {nhan[-2:]}")
print("Số giá trị:", len(ds["value"]), "| 5 giá trị đầu:", ds["value"][:5],
      "| kiểu:", type(ds["value"][0]).__name__)

3 ký tự cuối của tệp: '}}\x00' | số ký tự NUL: 1
Khoá cấp ngoài: ['dataset']
Khoá trong 'dataset': ['dimension', 'label', 'source', 'updated', 'value']
Tiêu đề bảng: Thu nhập bình quân một lao động đang làm việc phân theo địa phương chia theo Tỉnh/Thành phố và Năm
Chiều 'Tỉnh/Thành phố': 70 giá trị, ví dụ ['CẢ NƯỚC', 'Đồng bằng sông Hồng', 'Hà Nội', 'Vĩnh Phúc'] ... ['Bạc Liêu', 'Cà Mau']
Chiều 'Năm': 8 giá trị, ví dụ ['2018', '2019', '2020', '2021'] ... ['2024', 'Sơ bộ 2025']
Số giá trị: 560 | 5 giá trị đầu: [4849.53069148589, 5625.18150454766, 5445.0900593387, 5728.08937678263, 6639.42434549813] | kiểu: float


### c. Bảng tra

In [ ]:
# cấu trúc tệp JSON lồng nhiều cấp
tra_tho = json.loads(TEP_BANG_TRA.read_text(encoding="utf-8"))
print("Khoá cấp ngoài:", list(tra_tho))
tinh = tra_tho["DICT_PROVINCE"]
print("Số khoá trong DICT_PROVINCE:", len(tinh))
for khoa in list(tinh)[:3]:
    print(f"  {khoa!r} -> {tinh[khoa]}")

Khoá cấp ngoài: ['DICT_PROVINCE', 'DICT_PROVINCE_WARD_NO_DIVIDED', 'DICT_PROVINCE_WARD_DIVIDED']
Số khoá trong DICT_PROVINCE: 34
  'thanhphocantho' -> ['thanhphocantho', 'tinhhaugiang', 'tinhsoctrang']
  'thanhphodanang' -> ['thanhphodanang', 'tinhquangnam']
  'thanhphohaiphong' -> ['tinhhaiduong', 'thanhphohaiphong']


### Chọn cột cần dùng (tin rao)

| Cột | Giữ? | Lý do |
|---|---|---|
| `property_type_name` | có | Lọc riêng căn hộ (0.3) |
| `province_name` | có | Khoá ghép với thu nhập và bảng tra |
| `district_name`, `ward_name` | có | Thành phần của khoá nhận diện tin trùng |
| `price` | có | Tính giá/m² |
| `area` | có | Tính giá/m² |
| `bedroom_count` | có | Thành phần của khoá nhận diện tin trùng |
| `house_direction` | có | Chỉ dùng ở khảo sát để làm bằng chứng danh mục chưa chuẩn hoá |
| `published_at` | có | Cột ngày giờ (điều kiện 5), kiểm phân bố theo tháng |
| `name`, `description` | **không** | Văn bản tự do, có thể chứa thông tin cá nhân; không phục vụ câu hỏi |
| `street_name`, `project_name` | **không** | Chi tiết hơn mức tỉnh mà câu hỏi cần |
| `floor_count`, `frontage_width`, `house_depth`, `road_width`, `bathroom_count`, `balcony_direction` | **không** | Không phục vụ câu hỏi giá/m² theo tỉnh |

In [ ]:
COT_CAN = ["property_type_name", "province_name", "district_name", "ward_name",
           "price", "area", "bedroom_count", "house_direction", "published_at"]
LOAI_CAN_HO = "Căn hộ chung cư"

# Kiểm lại: mọi cột chọn đều có trong lược đồ, và loại căn hộ có trong dữ liệu
assert set(COT_CAN) <= set(bang_cot["cot"]), "Có cột không tồn tại trong lược đồ"
assert LOAI_CAN_HO in set(loai_nha["property_type_name"]), "Không thấy loại căn hộ"
print(f"Giữ {len(COT_CAN)}/{len(bang_cot)} cột; loại căn hộ = {LOAI_CAN_HO!r}")

Giữ 9/19 cột; loại căn hộ = 'Căn hộ chung cư'


## Phần 1 - Điều kiện 2: có tệp từ 5 MB trở lên

In [ ]:
bang = []
for p in sorted(THU_MUC_TIN_RAO.glob("shard_*.parquet")):
    meta = pq.ParquetFile(p).metadata
    bang.append({
        "tep": p.name,
        "kich_thuoc_MB": round(p.stat().st_size / 1e6, 1),
        "so_dong": meta.num_rows,
        "so_cot": meta.num_columns,
    })

for p in [TEP_THU_NHAP, TEP_BANG_TRA]:
    bang.append({"tep": p.name, "kich_thuoc_MB": round(p.stat().st_size / 1000000, 3)})

kich_thuoc = pd.DataFrame(bang)
print("Tổng dung lượng tin rao:",
      round(kich_thuoc.loc[kich_thuoc["tep"].str.startswith("shard"), "kich_thuoc_MB"].sum(), 1), "MB")
print("Tổng số dòng tin rao:", int(kich_thuoc["so_dong"].sum()))
print("Số tệp từ 5 MB trở lên:", int((kich_thuoc["kich_thuoc_MB"] >= 5).sum()))
kich_thuoc

Tổng dung lượng tin rao: 1669.2 MB
Tổng số dòng tin rao: 3500744
Số tệp từ 5 MB trở lên: 10


,tep,kich_thuoc_MB,so_dong,so_cot
0,shard_0000.parquet,163.700,350075.0,19.0
1,shard_0001.parquet,164.000,350075.0,19.0
2,shard_0002.parquet,162.700,350075.0,19.0
3,shard_0003.parquet,164.000,350075.0,19.0
4,shard_0004.parquet,165.600,350074.0,19.0
5,shard_0005.parquet,169.500,350074.0,19.0
6,shard_0006.parquet,169.600,350074.0,19.0
7,shard_0007.parquet,169.600,350074.0,19.0
8,shard_0008.parquet,171.500,350074.0,19.0
9,shard_0009.parquet,169.000,350074.0,19.0


**Kết luận:** 
* Cả **10/10** tệp Parquet đều lớn hơn 5 MB (mỗi tệp 162,7-171,5 MB)
* Hai tệp JSON nhỏ (0,012 MB và 0,596 MB)

## Phần 2 - Nạp ba nguồn

| Nguồn | Tệp | Định dạng | Vai trò trong câu hỏi |
|---|---|---|---|
| Tin rao bất động sản (TinixAI, Hugging Face) | `data/raw/tin_rao/shard_*.parquet` | Parquet, 10 phần | Giá và diện tích → giá/m² |
| Thu nhập bình quân một lao động (Cục Thống kê) | `data/raw/thu_nhap.json` | JSON-stat | Thu nhập → số năm để mua nhà |
| Bảng tra tỉnh cũ → tỉnh mới (`vietnamadminunits`) | `data/external/converter_2025.json` | JSON lồng nhiều cấp | Đưa kết quả lên 34 tỉnh mới |

Tin rao và thu nhập là **hai nguồn độc lập** (hai tổ chức khác nhau công bố, lược đồ khác nhau).
Bảng tra là bảng tham chiếu ngoài, đặt ở `data/external/`.

### a. Tin rao (Parquet)

Đọc cả thư mục một lần (pandas tự nối dọc 10 phần), chỉ lấy các cột trong `COT_CAN`.
Đo thời gian đọc để ước lượng rủi ro vượt giới hạn 5 phút của lệnh chạy toàn bộ.

In [ ]:
t0 = time.perf_counter()
tin_rao = pd.read_parquet(THU_MUC_TIN_RAO, columns=COT_CAN)
print(f"Đọc {len(tin_rao):,} dòng × {tin_rao.shape[1]} cột trong {time.perf_counter() - t0:.1f} giây")

Đọc 3,500,744 dòng × 9 cột trong 4.1 giây


### b. Thu nhập

JSON-stat gồm `dimension` (danh sách tỉnh, danh sách năm) và `value` (một mảng phẳng).
Giá trị xếp theo thứ tự **tỉnh × năm**: 8 giá trị đầu là 8 năm của tỉnh thứ nhất, rồi tới tỉnh
thứ hai, ... Đơn vị: **nghìn đồng/tháng**.

Ở đây dùng lại biến `js` đã đọc ở mục 0.4 (đã xử lý ký tự NUL thừa), không đọc tệp lần nữa.

In [ ]:
# Dùng lại biến `js` đã đọc ở 0.4 (đã bỏ ký tự NUL thừa), không đọc tệp lần nữa
ds = js["dataset"]


def nhan_theo_thu_tu(chieu: str) -> list[str]:
    """Lay nhan cua mot chieu JSON-stat theo dung thu tu vi tri (category.index)."""
    loai = ds["dimension"][chieu]["category"]
    return [loai["label"][k] for k, _ in sorted(loai["index"].items(), key=lambda kv: kv[1])]


tinh_tn = nhan_theo_thu_tu("Tỉnh/Thành phố")
nam_tn = nhan_theo_thu_tu("Năm")
assert len(ds["value"]) == len(tinh_tn) * len(nam_tn), "Số giá trị không khớp tỉnh × năm"

# JSON-stat xếp giá trị theo thứ tự các chiều trong ds["dimension"]["id"]: chiều cuối (Năm) chạy nhanh nhất
thu_nhap = pd.DataFrame([(t, n) for t in tinh_tn for n in nam_tn], columns=["tinh", "nam"])
thu_nhap["thu_nhap"] = ds["value"]
print("Thứ tự chiều:", ds["dimension"]["id"])
print(thu_nhap.shape)
# Kiểm thứ tự bằng mắt: Hà Nội và TP. Hồ Chí Minh phải cao hơn mức cả nước
thu_nhap[thu_nhap["tinh"].isin(["CẢ NƯỚC", "Hà Nội", "TP. Hồ Chí Minh"])].pivot(
    index="tinh", columns="nam", values="thu_nhap").round(0)

Thứ tự chiều: ['Tỉnh/Thành phố', 'Năm']
(560, 3)


nam,2018,2019,2020,2021,2022,2023,2024,Sơ bộ 2025
tinh,,,,,,,,
CẢ NƯỚC,4850.0,5625.0,5445.0,5728.0,6639.0,7087.0,7713.0,8426.0
Hà Nội,6395.0,7655.0,7090.0,7712.0,8859.0,9833.0,10815.0,12133.0
TP. Hồ Chí Minh,7859.0,8743.0,8486.0,7502.0,9103.0,9226.0,9843.0,10681.0


### 2c. Bảng tra tỉnh cũ --> tỉnh mới

`DICT_PROVINCE` có dạng `{slug_tỉnh_mới: [slug_tỉnh_cũ, ...]}`, viết không dấu, liền nhau
(ví dụ `thanhphohochiminh`). Đổi thành bảng hai cột, mỗi tỉnh cũ một dòng.

In [ ]:
bang_tra = pd.DataFrame(
    [(cu, moi) for moi, ds_cu in tra_tho["DICT_PROVINCE"].items() for cu in ds_cu],
    columns=["slug_cu", "slug_moi"],
)
print("Số tỉnh mới:", bang_tra["slug_moi"].nunique())
print("Số tỉnh cũ:", len(bang_tra))
print("slug_cu không trùng:", bang_tra["slug_cu"].is_unique)

Số tỉnh mới: 34
Số tỉnh cũ: 63
slug_cu không trùng: True


## Phần 3 - Tổng quan dữ liệu sau khi nạp

Bước đầu tiên của mọi lần khám phá dữ liệu: nhìn sơ qua từng bảng để biết **kích thước, kiểu dữ
liệu, vài dòng mẫu, số giá trị khác nhau và bộ nhớ** trước khi đi vào kiểm từng điều kiện.

### a. Tin rao

In [ ]:
print("Kích thước:", tin_rao.shape)
tin_rao.info(memory_usage="deep")
display(tin_rao.head())
display(tin_rao.sample(5, random_state=0))      # random_state cố định để chạy lại ra cùng mẫu
display(tin_rao.nunique().to_frame("so_gia_tri_khac_nhau"))
tin_rao.describe()

Kích thước: (3500744, 9)
<class 'pandas.DataFrame'>
RangeIndex: 3500744 entries, 0 to 3500743
Data columns (total 9 columns):
 #   Column              Dtype  
---  ------              -----  
 0   property_type_name  str    
 1   province_name       str    
 2   district_name       str    
 3   ward_name           str    
 4   price               str    
 5   area                float64
 6   bedroom_count       float64
 7   house_direction     str    
 8   published_at        str    
dtypes: float64(2), str(7)
memory usage: 501.5 MB


,property_type_name,province_name,district_name,ward_name,price,area,bedroom_count,house_direction,published_at
0,Nhà,Hà Nội,Bắc Từ Liêm,Phúc Diễn,7450000000,37.0,4.0,NaN,2025-06-01T05:12:56.941000
1,Nhà,Hồ Chí Minh,8,5,9100000000,48.0,3.0,NaN,2025-06-06T04:37:25.903000
2,Biệt thự/Nhà liền kề,Vĩnh Long,Bình Minh,NaN,20000000000,300.0,NaN,NaN,2025-06-15T18:58:07.455000
3,Căn hộ chung cư,Hồ Chí Minh,12,Đông Hưng Thuận,1300000000,44.0,1.0,NaN,2025-06-17T05:32:15.491000
4,Nhà,Hồ Chí Minh,Tân Phú,Tân Thành,9000000000,60.0,2.0,NaN,2025-06-06T04:06:17.378000


,property_type_name,province_name,district_name,ward_name,price,area,bedroom_count,house_direction,published_at
358441,Nhà,Hà Nội,Thanh Xuân,Thanh Xuân Bắc,21800000000,51.0,NaN,Tây Bắc,2025-07-04T17:10:26.111000
2463853,Biệt thự/Nhà liền kề,Hà Nội,Hà Đông,La Khê,47000000000,200.0,6.0,NaN,2025-12-13T14:18:56.377000
1680494,Nhà,Hồ Chí Minh,Hóc Môn,NaN,4000000000,88.1,2.0,Tây Bắc,2025-10-15T15:05:56.364000
1148456,Nhà,Hà Nội,Nam Từ Liêm,Mỹ Đình 2,7980000000,37.0,4.0,NaN,2025-09-18T07:53:09.403000
984215,Đất,Hà Nội,Sóc Sơn,Quang Tiến,3000000000,80.0,NaN,Đông - Nam,2025-08-23T09:33:44.609000


,so_gia_tri_khac_nhau
property_type_name,5
province_name,63
district_name,626
ward_name,4416
price,34132
area,24525
bedroom_count,110
house_direction,31
published_at,3497379


,area,bedroom_count
count,3.500740e+06,1.900383e+06
mean,5.050264e+02,3.373194e+00
std,9.726008e+04,2.713995e+00
min,1.000000e+00,1.000000e+00
25%,5.600000e+01,2.000000e+00
50%,8.000000e+01,3.000000e+00
75%,1.200000e+02,4.000000e+00
max,8.200000e+07,2.000000e+02


### b. Thu nhập

In [ ]:
print("Kích thước:", thu_nhap.shape)
print(thu_nhap.dtypes, "\n")
print("Số địa danh:", thu_nhap["tinh"].nunique(), "| Các năm:", list(thu_nhap["nam"].unique()))
thu_nhap.head(10)

Kích thước: (560, 3)
tinh            str
nam             str
thu_nhap    float64
dtype: object 

Số địa danh: 70 | Các năm: ['2018', '2019', '2020', '2021', '2022', '2023', '2024', 'Sơ bộ 2025']


,tinh,nam,thu_nhap
0,CẢ NƯỚC,2018,4849.530691
1,CẢ NƯỚC,2019,5625.181505
2,CẢ NƯỚC,2020,5445.090059
3,CẢ NƯỚC,2021,5728.089377
4,CẢ NƯỚC,2022,6639.424345
5,CẢ NƯỚC,2023,7086.602006
6,CẢ NƯỚC,2024,7713.432090
7,CẢ NƯỚC,Sơ bộ 2025,8426.373797
8,Đồng bằng sông Hồng,2018,5314.394276
9,Đồng bằng sông Hồng,2019,6378.656443


### c. Bảng tra

In [ ]:
print("Kích thước:", bang_tra.shape)
print(bang_tra.dtypes, "\n")
display(bang_tra.head(10))
so_tinh_cu = bang_tra.groupby("slug_moi").size().sort_values(ascending=False)
print("Số tỉnh cũ gộp vào mỗi tỉnh mới:")
print(so_tinh_cu.value_counts().sort_index().rename("so_tinh_moi").to_string())
vi_du = so_tinh_cu.index[0]
print("Ví dụ:", vi_du, "<-", bang_tra.loc[bang_tra["slug_moi"] == vi_du, "slug_cu"].tolist())

Kích thước: (63, 2)
slug_cu     str
slug_moi    str
dtype: object 



,slug_cu,slug_moi
0,thanhphocantho,thanhphocantho
1,tinhhaugiang,thanhphocantho
2,tinhsoctrang,thanhphocantho
3,thanhphodanang,thanhphodanang
4,tinhquangnam,thanhphodanang
5,tinhhaiduong,thanhphohaiphong
6,thanhphohaiphong,thanhphohaiphong
7,thanhphohanoi,thanhphohanoi
8,tinhbinhduong,thanhphohochiminh
9,tinhbariavungtau,thanhphohochiminh


Số tỉnh cũ gộp vào mỗi tỉnh mới:
1    11
2    17
3     6
Ví dụ: thanhphocantho <- ['thanhphocantho', 'tinhhaugiang', 'tinhsoctrang']


**Nhận xét:**
- **Tin rao:** 3.500.744 dòng × 9 cột đã chọn, đọc trong chưa tới 1 giây, chiếm khoảng 500 MB bộ nhớ.
  + `price` và `published_at` đang là **chuỗi** dù mang ý nghĩa số và ngày (→ Phần 6.3).
  + `house_direction` có 31 giá trị khác nhau trong khi thực tế chỉ có 8 hướng (→ Phần 6.4).
  + `area` lớn nhất 82 triệu m² (→ Phần 6.6).
- **Thu nhập:** 560 dòng = 70 địa danh × 8 năm (2018 → "Sơ bộ 2025"), đơn vị nghìn đồng/tháng.
  + Thứ tự làm phẳng JSON-stat đã kiểm: Hà Nội và TP.HCM cao hơn mức cả nước ở mọi năm.
  + Cột năm là chuỗi; 70 địa danh gồm cả dòng tổng hợp (→ Phần 6.3, 6.4).
- **Bảng tra:** 63 tỉnh cũ → 34 tỉnh mới, không trùng khoá; 11 tỉnh giữ nguyên, 17 tỉnh gộp từ 2,
  + 6 tỉnh gộp từ 3. Khoá là chuỗi không dấu có tiền tố (`tinhbinhduong`) → khác cách viết của hai nguồn kia (→ Phần 7).

## Phần 4 - Điều kiện 5: có cột ngày hoặc giờ

**Mục tiêu:** chứng minh `published_at` (ngày đăng tin) đọc được thành kiểu ngày giờ, và chỉ ra
khoảng thời gian dữ liệu bao phủ.

In [ ]:
print("Kiểu ban đầu của published_at:", tin_rao["published_at"].dtype)
ngay = pd.to_datetime(tin_rao["published_at"], format="ISO8601", errors="coerce")
print("Số ô không ép được sang ngày giờ:", int(ngay.isna().sum()))
print("Ngày sớm nhất:", ngay.min())
print("Ngày muộn nhất:", ngay.max())

Kiểu ban đầu của published_at: str
Số ô không ép được sang ngày giờ: 0
Ngày sớm nhất: 2025-06-01 00:01:38.762000
Ngày muộn nhất: 2026-03-30 23:59:56.217000


In [ ]:
thang = ngay.dt.to_period("M")
display(thang.value_counts().sort_index().to_frame("so_tin"))

# Khoảng ngày của từng phần Parquet (chỉ đọc cột published_at của từng phần)
khoang = []
for p in cac_phan:
    ngay_phan = pd.to_datetime(pd.read_parquet(p, columns=["published_at"])["published_at"],
                               format="ISO8601", errors="coerce")
    khoang.append({"tep": p.name, "tu_ngay": ngay_phan.min().date(), "den_ngay": ngay_phan.max().date()})
pd.DataFrame(khoang)

,so_tin
published_at,
2025-06,257726
2025-07,364215
2025-08,365761
2025-09,352239
2025-10,415170
2025-11,407202
2025-12,444361
2026-01,283760
2026-02,186331


,tep,tu_ngay,den_ngay
0,shard_0000.parquet,2025-06-01,2025-07-30
1,shard_0001.parquet,2025-07-01,2025-08-30
2,shard_0002.parquet,2025-08-01,2025-09-29
3,shard_0003.parquet,2025-09-01,2025-10-30
4,shard_0004.parquet,2025-10-01,2025-10-30
5,shard_0005.parquet,2025-10-01,2025-11-29
6,shard_0006.parquet,2025-11-01,2025-12-30
7,shard_0007.parquet,2025-12-01,2026-01-30
8,shard_0008.parquet,2026-01-01,2026-03-30
9,shard_0009.parquet,2026-03-01,2026-03-30


**Kết luận:** `published_at` ban đầu là chuỗi, ép sang ngày giờ **không lỗi ô nào** (0 ô không ép
được). Dữ liệu trải từ **01/06/2025 đến 30/03/2026** (10 tháng, mỗi tháng từ 186 nghìn đến 444 nghìn
tin). Mỗi phần Parquet ứng với một khoảng thời gian liên tiếp → không được bỏ bớt phần nào khi chạy.
→ **ĐẠT** (vế ngày giờ của điều kiện 5)

## Phần 5 - Điều kiện 5: giấy phép cho phép dùng cho mục đích học tập

Phần này không cần code: giấy phép là văn bản do nơi công bố dữ liệu đưa ra, được chép lại và
trích nguyên văn dưới đây.

| Nguồn | Nơi công bố | Giấy phép | Câu cho phép (trích nguyên văn) | Điều kiện kèm theo |
|---|---|---|---|---|
| Tin rao bất động sản `tinixai/vietnam-real-estates` | TinixAI, https://huggingface.co/datasets/tinixai/vietnam-real-estates | **CC BY-NC 4.0** | "Free to use for academic research and education." | Ghi nguồn; không dùng thương mại ("Commercial use requires explicit written permission from TiniX AI.") |
| Thu nhập bình quân một lao động đang làm việc phân theo địa phương, bảng V14.36 | Cục Thống kê, https://pxweb.nso.gov.vn/pxweb/vi/ | Dữ liệu công bố công khai - trường **"Bản quyền: Không"** trên bảng PX-Web | Không giới hạn bản quyền trên bảng | Ghi rõ nguồn: trang Thông tin điện tử Cục Thống kê (www.nso.gov.vn) |
| Bảng tra `converter_2025.json` của gói `vietnamadminunits` | Trần Ngọc Minh Hiếu, https://github.com/tranngocminhhieu/vietnamadminunits | **MIT** | "Permission is hereby granted, free of charge, to any person obtaining a copy of this software …" | Giữ thông báo bản quyền MIT khi phân phối lại |

Đồ án là bài tập học phần, phi thương mại, có ghi nguồn đầy đủ → phù hợp cả ba giấy phép.

**Không có dữ liệu cá nhân** (mục "Chú ý" ở 3.11 của đề):
- Nguồn tin rao đã xoá số điện thoại trong tiêu đề và mô tả ("phone numbers redacted").
- Nhóm **không nạp** hai cột văn bản tự do `name` và `description`, nên không có họ tên hay địa chỉ
  nhà riêng nào đi vào chương trình.
- Thu nhập và bảng tra là số liệu tổng hợp theo tỉnh, không có cá nhân.
- Không nguồn nào được lấy bằng cách vượt cơ chế chặn truy cập: tin rao tải bằng công cụ chính thức
  của Hugging Face, thu nhập xuất bằng nút xuất của PX-Web, bảng tra tải từ GitHub công khai.

**Kết luận:** cả ba nguồn đều cho phép dùng cho mục đích học tập → **ĐẠT** (kết hợp Phần 4 cho vế ngày giờ).

## Phần 6 - Điều kiện 4: ít nhất ba trong bốn khuyết tật

**Mục tiêu:** đếm từng loại khuyết tật. Mỗi loại ghi rõ **định nghĩa cách đếm** để giải thích được
khi vấn đáp.

### a. Giá trị thiếu

**Cách đếm:** số ô rỗng (`NaN`) theo từng cột. Ngoài ra đếm `price == 0`: tin "giá thoả thuận"
có thể bị ghi thành 0 thay vì để trống - đây là **thiếu trá hình**, nếu không xử lý sẽ kéo trung vị
giá xuống.

In [ ]:
# price được lưu dạng CHUỖI (thấy ở 0.1) -> đổi sang số trước khi đếm. Chi tiết ở 6.3.
gia_chuoi = tin_rao["price"]
tin_rao["price"] = pd.to_numeric(gia_chuoi, errors="coerce")
print("price: kiểu gốc", gia_chuoi.dtype, "-> kiểu mới", tin_rao["price"].dtype,
      "| chuỗi không đổi được sang số:", int((tin_rao["price"].isna() & gia_chuoi.notna()).sum()))

can_ho = tin_rao[tin_rao["property_type_name"] == LOAI_CAN_HO]

thieu = pd.DataFrame({"so_o_thieu": tin_rao.isna().sum()})
thieu["ty_le_%"] = (thieu["so_o_thieu"] / len(tin_rao) * 100).round(2)
thieu["can_ho_so_o_thieu"] = can_ho.isna().sum()
thieu["can_ho_ty_le_%"] = (thieu["can_ho_so_o_thieu"] / len(can_ho) * 100).round(2)
display(thieu)

gia_bang_0 = tin_rao["price"].eq(0)
print("Số tin price == 0:", int(gia_bang_0.sum()),
      "| riêng căn hộ:", int(can_ho["price"].eq(0).sum()))
print("Tỷ lệ dòng có ít nhất một ô thiếu (9 cột đã nạp):",
      f"{tin_rao.isna().any(axis=1).mean() * 100:.1f}%")

# Giá trống (NaN) và giá bằng 0 theo tháng: cách ghi "thiếu giá" có đổi theo thời gian không
pd.DataFrame({"price_trong": tin_rao["price"].isna(), "price_bang_0": gia_bang_0}).groupby(thang).sum()

price: kiểu gốc str -> kiểu mới float64 | chuỗi không đổi được sang số: 0


,so_o_thieu,ty_le_%,can_ho_so_o_thieu,can_ho_ty_le_%
property_type_name,0,0.00,0,0.00
province_name,0,0.00,0,0.00
district_name,101933,2.91,43602,5.72
ward_name,496716,14.19,159141,20.86
price,218494,6.24,63015,8.26
area,4,0.00,2,0.00
bedroom_count,1600361,45.71,52917,6.94
house_direction,2414214,68.96,432623,56.72
published_at,0,0.00,0,0.00


Số tin price == 0: 57320 | riêng căn hộ: 21890
Tỷ lệ dòng có ít nhất một ô thiếu (9 cột đã nạp): 85.6%


,price_trong,price_bang_0
published_at,,
2025-06,7510,0
2025-07,21555,0
2025-08,36066,0
2025-09,32697,0
2025-10,34772,0
2025-11,34245,0
2025-12,37893,0
2026-01,13751,11945
2026-02,4,13464


### b. Bản ghi trùng

**Cách đếm:** trùng hoàn toàn hiếm khi xảy ra vì mỗi lần đăng lại có giờ đăng khác. Một tin đăng
lại nhiều lần thường giống nhau ở vị trí, giá, diện tích, số phòng ngủ → định nghĩa trùng theo
khoá `KHOA_TRUNG` dưới đây.

In [ ]:
KHOA_TRUNG = ["province_name", "district_name", "ward_name", "price", "area", "bedroom_count"]

print("(a) Trùng trên toàn bộ 9 cột đã nạp:", int(tin_rao.duplicated().sum()))

# (b) chỉ xét tin có giá và diện tích dương: nếu không, các tin cùng thiếu giá sẽ bị coi là "trùng"
co_gia = tin_rao[(tin_rao["price"] > 0) & (tin_rao["area"] > 0)]
print("(b) Trùng theo KHOA_TRUNG (tin có giá và diện tích):", int(co_gia.duplicated(subset=KHOA_TRUNG).sum()),
      "/", len(co_gia))
can_ho_co_gia = co_gia[co_gia["property_type_name"] == LOAI_CAN_HO]
print("    riêng căn hộ:", int(can_ho_co_gia.duplicated(subset=KHOA_TRUNG).sum()), "/", len(can_ho_co_gia))

# (c) tiêu đề lặp lại: đọc riêng cột name chỉ để đếm, không in nội dung
tieu_de = pd.read_parquet(THU_MUC_TIN_RAO, columns=["name"])["name"]
print("(c) Tiêu đề lặp lại:", int(tieu_de.duplicated().sum()), "/", len(tieu_de))
del tieu_de

(a) Trùng trên toàn bộ 9 cột đã nạp: 113
(b) Trùng theo KHOA_TRUNG (tin có giá và diện tích): 1551238 / 3224930
    riêng căn hộ: 351669 / 677895
(c) Tiêu đề lặp lại: 961363 / 3500744


### c. Kiểu dữ liệu sai

**Cách đếm:** cột mang ý nghĩa ngày hoặc số nhưng lưu ở kiểu khác, hoặc lẫn nhiều khuôn dạng.

Đáng chú ý: thẻ dữ liệu trên Hugging Face ghi `price` là `float64`, nhưng tệp Parquet thật lưu
`price` dạng **chuỗi** (lược đồ ở 0.1). Chỉ tin vào thẻ dữ liệu mà không kiểm thì sẽ sai.

In [ ]:
print("price — kiểu gốc:", gia_chuoi.dtype, "| số ô có giá trị:", int(gia_chuoi.notna().sum()),
      "| ví dụ:", gia_chuoi.dropna().head(3).tolist())

pa = tin_rao["published_at"]
co_phan_le = pa.str.contains(".", regex=False)
print("published_at — kiểu:", pa.dtype,
      "| có phần lẻ giây:", int(co_phan_le.sum()), "| không có:", int((~co_phan_le).sum()))
print("   ví dụ không có phần lẻ giây:", pa[~co_phan_le].head(3).tolist())

chi_so = tin_rao["district_name"].str.fullmatch(r"\d+", na=False)
print("district_name chỉ gồm chữ số:", int(chi_so.sum()),
      "| ví dụ:", tin_rao.loc[chi_so, "district_name"].value_counts().head(5).index.tolist())

print("Cột năm của bảng thu nhập — kiểu:", thu_nhap["nam"].dtype,
      "| giá trị cuối:", thu_nhap["nam"].unique()[-2:].tolist())

price — kiểu gốc: str | số ô có giá trị: 3282250 | ví dụ: ['7450000000', '9100000000', '20000000000']
published_at — kiểu: str | có phần lẻ giây: 3478720 | không có: 22024
   ví dụ không có phần lẻ giây: ['2025-06-24T05:29:17', '2025-06-20T10:12:22', '2025-06-12T23:53:44']
district_name chỉ gồm chữ số: 417666 | ví dụ: ['7', '12', '1', '3', '8']
Cột năm của bảng thu nhập — kiểu: str | giá trị cuối: ['2024', 'Sơ bộ 2025']


### d. Danh mục chưa chuẩn hoá

**Cách đếm:** cùng một giá trị nhưng viết nhiều cách, hoặc lẫn giá trị không cùng cấp.

In [ ]:
display(tin_rao["house_direction"].value_counts().to_frame("so_tin"))

# Dòng tổng hợp lẫn trong bảng thu nhập: cả nước + 6 vùng kinh tế (thấy ở 0.4)
TONG_HOP = ["CẢ NƯỚC", "Đồng bằng sông Hồng", "Trung du và miền núi phía Bắc",
            "Bắc Trung Bộ và duyên hải miền Trung", "Tây Nguyên", "Đông Nam Bộ",
            "Đồng bằng sông Cửu Long"]
assert set(TONG_HOP) <= set(thu_nhap["tinh"]), "Tên dòng tổng hợp không khớp"
print("Địa danh trong bảng thu nhập:", thu_nhap["tinh"].nunique(),
      "= 7 dòng tổng hợp +", thu_nhap["tinh"].nunique() - len(TONG_HOP), "tỉnh")

# Tên tỉnh viết khác nhau giữa hai nguồn (so khớp nguyên chuỗi, chưa chuẩn hoá)
ten_tin_rao = set(tin_rao["province_name"].unique())
ten_thu_nhap = set(thu_nhap["tinh"]) - set(TONG_HOP)
print("Có trong tin rao, không có trong thu nhập:", sorted(ten_tin_rao - ten_thu_nhap))
print("Có trong thu nhập, không có trong tin rao:", sorted(ten_thu_nhap - ten_tin_rao))
print("Ví dụ khoá ở bảng tra:", bang_tra["slug_cu"].head(3).tolist())

,so_tin
house_direction,
Đông Nam,145317
Nam,118194
Đông,105773
Đông - Nam,100556
Tây Bắc,99547
Đông Bắc,92433
Bắc,83539
Tây Nam,79100
Tây - Bắc,73043


Địa danh trong bảng thu nhập: 70 = 7 dòng tổng hợp + 63 tỉnh
Có trong tin rao, không có trong thu nhập: ['Hồ Chí Minh']
Có trong thu nhập, không có trong tin rao: ['TP. Hồ Chí Minh']
Ví dụ khoá ở bảng tra: ['thanhphocantho', 'tinhhaugiang', 'tinhsoctrang']


### e. Danh mục tỉnh: tên tỉnh cũ có bị lẫn tên tỉnh mới không

Tin rao kéo dài qua mốc 01/7/2025 (sáp nhập tỉnh). Nếu nguồn bắt đầu ghi tin ở Bình Dương thành
"Hồ Chí Minh", cột `province_name` sẽ lẫn **hai hệ danh mục** (63 tỉnh cũ và 34 tỉnh mới). Khi đó giá
sẽ bị ghép nhầm với thu nhập của TP.HCM cũ mà code không báo lỗi.

**Cách kiểm:** theo từng tháng, tỷ lệ tin của các tỉnh cũ đã bị sáp nhập. Nếu tỷ lệ sụt mạnh sau
07/2025 trong khi tỉnh nhận tăng lên → có lẫn.

In [ ]:
TINH_THEO_DOI = ["Bình Dương", "Bà Rịa - Vũng Tàu", "Hải Dương", "Hồ Chí Minh", "Hải Phòng"]
assert set(TINH_THEO_DOI) <= set(tin_rao["province_name"]), "Tên tỉnh theo dõi không có trong dữ liệu"
ty_le_thang = pd.crosstab(thang, tin_rao["province_name"], normalize="index")[TINH_THEO_DOI]
(ty_le_thang * 100).round(2)

province_name,Bình Dương,Bà Rịa - Vũng Tàu,Hải Dương,Hồ Chí Minh,Hải Phòng
published_at,,,,,
2025-06,4.46,1.46,0.11,34.11,2.21
2025-07,4.64,1.39,0.05,33.90,2.03
2025-08,4.65,1.38,0.06,33.31,2.24
2025-09,4.30,1.43,0.09,34.44,2.11
2025-10,4.01,1.21,0.10,40.34,2.01
2025-11,3.75,1.36,0.11,39.47,2.15
2025-12,4.23,1.18,0.07,42.25,1.97
2026-01,4.23,1.24,0.06,45.28,1.93
2026-02,3.69,1.35,0.06,41.90,2.07


### f. Giá trị bất thường

In [ ]:
can_ho_co_gia = can_ho[(can_ho["price"] > 0) & (can_ho["area"] > 0)]
gia_m2 = can_ho_co_gia["price"] / can_ho_co_gia["area"] / 1e6        # triệu đồng/m²
phan_vi = pd.DataFrame({
    "area_m2": can_ho_co_gia["area"].quantile([0, 0.01, 0.5, 0.99, 1]),
    "price_ty_dong": (can_ho_co_gia["price"] / 1e9).quantile([0, 0.01, 0.5, 0.99, 1]),
    "gia_trieu_m2": gia_m2.quantile([0, 0.01, 0.5, 0.99, 1]),
}).round(2)
display(phan_vi)
print("Căn hộ có area < 10 m²:", int((can_ho_co_gia["area"] < 10).sum()),
      "| area > 1000 m²:", int((can_ho_co_gia["area"] > 1000).sum()))
print("Căn hộ có giá/m² < 5 triệu:", int((gia_m2 < 5).sum()),
      "| > 500 triệu:", int((gia_m2 > 500).sum()))

,area_m2,price_ty_dong,gia_trieu_m2
0.00,3.0,0.0,0.00
0.01,30.0,1.1,21.14
0.50,73.0,4.9,68.00
0.99,235.0,37.0,261.86
1.00,82000000.0,999910.0,348571.43


Căn hộ có area < 10 m²: 15 | area > 1000 m²: 56
Căn hộ có giá/m² < 5 triệu: 576 | > 500 triệu: 921


### Bảng tổng hợp khuyết tật

| Khuyết tật | Cách đếm | Con số | Có? |
|---|---|---|---|
| Giá trị thiếu | Ô rỗng theo cột; `price == 0` | `price` trống 218.494 ô (6,24%); `ward_name` 496.716 (14,19%); `district_name` 101.933 (2,91%); thêm 57.320 tin `price == 0` | có |
| Bản ghi trùng | Trùng toàn bộ cột; trùng theo `KHOA_TRUNG`; tiêu đề lặp | Toàn bộ 9 cột: 113; theo khoá (tin có giá): 1.551.238 / 3.224.930; tiêu đề lặp: 961.363 | có |
| Kiểu dữ liệu sai | `price` là chuỗi; `published_at` là chuỗi, 2 khuôn dạng; `district_name` chỉ có số; năm là chuỗi | `price` chuỗi ở cả 3.282.250 ô; 22.024 dòng ngày giờ thiếu phần lẻ giây; 417.666 dòng quận chỉ là số | có |
| Danh mục chưa chuẩn hoá | `house_direction` nhiều cách viết; dòng tổng hợp trong bảng thu nhập; tên tỉnh khác nhau giữa 3 nguồn | 31 giá trị hướng (vd "Đông Nam" 145.317 và "Đông - Nam" 100.556); 7 dòng tổng hợp; "Hồ Chí Minh" ≠ "TP. Hồ Chí Minh" ≠ `thanhphohochiminh` | có |

**Kết luận:** có đủ **4/4** loại khuyết tật → **ĐẠT** (đề chỉ cần 3/4).

**Ghi chú quan trọng cho khâu làm sạch:**
- **Cách ghi "thiếu giá" đổi theo thời gian (6.1):** từ 06/2025 đến 12/2025 giá thiếu để **trống**
  (0 tin `price == 0`); từ 01/2026 chuyển sang ghi **0** (02/2026: 4 ô trống nhưng 13.464 ô bằng 0).
  → `price == 0` phải coi là thiếu, nếu không trung vị các tháng 2026 bị kéo xuống.
- **Không dùng `dropna()` chung:** 85,6% dòng có ít nhất một ô thiếu (chủ yếu do `house_direction`
  thiếu 68,96%, `bedroom_count` thiếu 45,71%) → bỏ dòng theo từng cột cần cho câu hỏi.
- **Trùng theo khoá rất cao** (~48% tin có giá; ~52% căn hộ): khoá hiện tại có thể **quá lỏng** — các căn
  khác nhau trong cùng dự án, cùng giá, cùng diện tích, cùng số phòng ngủ sẽ bị coi là trùng. Đây là
  **cận trên** của số tin trùng; khâu làm sạch cần thêm cột vào khoá (ví dụ `project_name`,
  `published_at` theo ngày) và so sánh lại.
- **Rủi ro lẫn tên tỉnh (6.5): không thấy.** Tỷ lệ tin Bình Dương giữ ổn định 3,7-4,8% mỗi tháng,
  Bà Rịa - Vũng Tàu 1,2-1,5%, Hải Dương 0,05-0,11%, không sụt sau 07/2025. Tỷ lệ TP.HCM tăng từ
  khoảng 34% lên 40-45% từ 10/2025 nhưng không đi kèm tỉnh cũ nào giảm → do số tin TP.HCM tăng,
  không phải do đổi tên.
- **Giá trị bất thường (6.6, căn hộ):** diện tích 3 m² → 82 triệu m²; giá/m² trung vị 68 triệu, phân vị
  1%-99% là 21-262 triệu/m²; 576 căn dưới 5 triệu/m² và 921 căn trên 500 triệu/m² → cần ngưỡng lọc
  (để trong `cau_hinh.yaml`).

## Phần 7 - Điều kiện 1: hai nguồn độc lập ghép được bằng khoá

**Logic ghép:**

```
tin_rao (nhiều tin mỗi tỉnh)
   ── merge theo tỉnh cũ ──▶ thu_nhap năm "Sơ bộ 2025" (1 dòng mỗi tỉnh)
   ── merge theo tỉnh cũ ──▶ bang_tra (1 dòng mỗi tỉnh cũ) → thêm tỉnh mới
```

Cả hai phép ghép đều **nhiều-một** (`validate="many_to_one"`), nên số dòng sau ghép phải **bằng**
số dòng trước ghép. Nếu tăng lên nghĩa là bảng bên phải có tỉnh bị trùng; nếu có tin không nhận
được thu nhập nghĩa là có khoá không khớp.

**Vướng mắc:** tên tỉnh viết khác nhau giữa ba nguồn ("Hồ Chí Minh", "TP. Hồ Chí Minh",
`thanhphohochiminh`) → phải đưa về cùng một khoá bằng hàm `slug()` trước khi ghép.

In [ ]:
def slug(ten: str) -> str:
    """Dua ten tinh ve dang khong dau, viet thuong, bo tien to hanh chinh."""
    khong_dau = unicodedata.normalize("NFD", ten)
    khong_dau = "".join(k for k in khong_dau if unicodedata.category(k) != "Mn")
    khong_dau = khong_dau.replace("đ", "d").replace("Đ", "D").lower()
    chi_chu = re.sub(r"[^a-z]", "", khong_dau)
    return re.sub(r"^(thanhpho|tinh|tp)", "", chi_chu)


for vi_du in ["Hồ Chí Minh", "TP. Hồ Chí Minh", "thanhphohochiminh", "Bà Rịa - Vũng Tàu",
              "tinhbariavungtau", "Huế", "Đắk Lắk"]:
    print(f"{vi_du!r:>24} -> {slug(vi_du)!r}")

           'Hồ Chí Minh' -> 'hochiminh'
       'TP. Hồ Chí Minh' -> 'hochiminh'
     'thanhphohochiminh' -> 'hochiminh'
     'Bà Rịa - Vũng Tàu' -> 'bariavungtau'
      'tinhbariavungtau' -> 'bariavungtau'
                   'Huế' -> 'hue'
               'Đắk Lắk' -> 'daklak'


In [ ]:
# Áp slug() cho từng tên khác nhau rồi ánh xạ (nhanh hơn gọi slug() trên 3,5 triệu dòng)
ten_rao = tin_rao["province_name"].unique()
tin_rao["tinh_cu"] = tin_rao["province_name"].map({t: slug(t) for t in ten_rao})

thu_nhap_tinh = thu_nhap[~thu_nhap["tinh"].isin(TONG_HOP)].copy()
thu_nhap_tinh["tinh_cu"] = thu_nhap_tinh["tinh"].map(slug)

bang_tra["tinh_cu"] = bang_tra["slug_cu"].map(slug)
# Tên tỉnh mới dễ đọc: mỗi tỉnh mới mang tên của một tỉnh cũ, lấy từ bảng thu nhập
ten_doc_duoc = dict(zip(thu_nhap_tinh["tinh_cu"], thu_nhap_tinh["tinh"]))
bang_tra["tinh_moi"] = bang_tra["slug_moi"].map(slug).map(ten_doc_duoc)
assert bang_tra["tinh_moi"].notna().all(), "Có tỉnh mới không tìm được tên"

khoa_rao = set(tin_rao["tinh_cu"])
khoa_tn = set(thu_nhap_tinh["tinh_cu"])
khoa_tra = set(bang_tra["tinh_cu"])
print("Số khoá — tin rao:", len(khoa_rao), "| thu nhập:", len(khoa_tn), "| bảng tra:", len(khoa_tra))
print("Tin rao − thu nhập:", sorted(khoa_rao - khoa_tn))
print("Thu nhập − tin rao:", sorted(khoa_tn - khoa_rao))
print("Tin rao − bảng tra:", sorted(khoa_rao - khoa_tra))
print("Bảng tra − tin rao:", sorted(khoa_tra - khoa_rao))

# Bảng đối chiếu tên gốc của cả ba nguồn theo khoá chung
doi_chieu = (pd.DataFrame({"tinh_cu": sorted(khoa_rao | khoa_tn | khoa_tra)})
             .merge(tin_rao[["tinh_cu", "province_name"]].drop_duplicates(), how="left")
             .merge(thu_nhap_tinh[["tinh_cu", "tinh"]].drop_duplicates(), how="left")
             .merge(bang_tra[["tinh_cu", "slug_cu", "tinh_moi"]], how="left"))
doi_chieu

Số khoá — tin rao: 63 | thu nhập: 63 | bảng tra: 63
Tin rao − thu nhập: []
Thu nhập − tin rao: []
Tin rao − bảng tra: []
Bảng tra − tin rao: []


,tinh_cu,province_name,tinh,slug_cu,tinh_moi
0,angiang,An Giang,An Giang,tinhangiang,An Giang
1,bacgiang,Bắc Giang,Bắc Giang,tinhbacgiang,Bắc Ninh
2,backan,Bắc Kạn,Bắc Kạn,tinhbackan,Thái Nguyên
3,baclieu,Bạc Liêu,Bạc Liêu,tinhbaclieu,Cà Mau
4,bacninh,Bắc Ninh,Bắc Ninh,tinhbacninh,Bắc Ninh
...,...,...,...,...,...
58,travinh,Trà Vinh,Trà Vinh,tinhtravinh,Vĩnh Long
59,tuyenquang,Tuyên Quang,Tuyên Quang,tinhtuyenquang,Tuyên Quang
60,vinhlong,Vĩnh Long,Vĩnh Long,tinhvinhlong,Vĩnh Long
61,vinhphuc,Vĩnh Phúc,Vĩnh Phúc,tinhvinhphuc,Phú Thọ


In [ ]:
tn25 = thu_nhap_tinh.loc[thu_nhap_tinh["nam"] == "Sơ bộ 2025", ["tinh_cu", "thu_nhap"]]
assert tn25["tinh_cu"].is_unique

g = tin_rao.merge(tn25, on="tinh_cu", how="left", validate="many_to_one", indicator="ghep_1")
g = g.merge(bang_tra[["tinh_cu", "tinh_moi"]], on="tinh_cu", how="left",
            validate="many_to_one", indicator="ghep_2")

print("Số dòng trước ghép:", len(tin_rao), "| sau ghép:", len(g), "| bằng nhau:", len(tin_rao) == len(g))
print("Ghép 1 (thu nhập):", g["ghep_1"].value_counts().to_dict())
print("Ghép 2 (bảng tra):", g["ghep_2"].value_counts().to_dict())
print("Số tỉnh mới sau ghép:", g["tinh_moi"].nunique())
display(g[["province_name", "tinh_cu", "thu_nhap", "tinh_moi"]].drop_duplicates().head())
del g

Số dòng trước ghép: 3500744 | sau ghép: 3500744 | bằng nhau: True
Ghép 1 (thu nhập): {'both': 3500744, 'left_only': 0, 'right_only': 0}
Ghép 2 (bảng tra): {'both': 3500744, 'left_only': 0, 'right_only': 0}
Số tỉnh mới sau ghép: 34


,province_name,tinh_cu,thu_nhap,tinh_moi
0,Hà Nội,hanoi,12133.151470,Hà Nội
1,Hồ Chí Minh,hochiminh,10680.679259,TP. Hồ Chí Minh
2,Vĩnh Long,vinhlong,6739.745422,Vĩnh Long
5,Đà Nẵng,danang,9828.875517,Đà Nẵng
7,Khánh Hòa,khanhhoa,7699.540075,Khánh Hòa


**Kết luận:** sau khi chuẩn hoá bằng `slug()`, cả ba nguồn có đúng **63 khoá tỉnh cũ, khớp hoàn toàn**
(0 khoá không khớp ở mọi cặp). Trước chuẩn hoá chỉ lệch đúng một tỉnh giữa tin rao và thu nhập
("Hồ Chí Minh" / "TP. Hồ Chí Minh"), còn bảng tra thì lệch toàn bộ vì viết dạng `tinh...`/`thanhpho...`.
Ghép thử trên toàn bộ 3.500.744 tin: số dòng **trước = sau**, 0 dòng `left_only` ở cả hai phép ghép,
ra đủ **34 tỉnh mới**. Hai nguồn độc lập (TinixAI và Cục Thống kê) hợp nhất được bằng mã → **ĐẠT**

## Phần 8 - Điều kiện 3: từ 20.000 dòng trở lên sau hợp nhất

Nhóm chỉ có **ba bảng** (tin rao, thu nhập, bảng tra), ít hơn bốn, nên **bắt buộc** phải đạt từ
20.000 dòng trở lên sau khi hợp nhất.

**Cách đếm:** đếm số dòng còn lại qua từng bước lọc nháp ("phễu"), rồi sau khi ghép.
Bộ lọc nháp: căn hộ → `price > 0` → `area > 0` → bỏ trùng theo `KHOA_TRUNG` → ghép.
Đây chỉ là bộ lọc thô để ước lượng; khâu làm sạch thật (tuần 13) sẽ chặt hơn.

In [ ]:
NGUONG_IT_TIN = 30          # ngưỡng nháp; khâu phân tích thật sẽ đặt trong cau_hinh.yaml

buoc = [("Toàn bộ tin rao", len(tin_rao))]
d = tin_rao[tin_rao["property_type_name"] == LOAI_CAN_HO]
buoc.append(("Chỉ căn hộ", len(d)))
d = d[d["price"] > 0]
buoc.append(("price > 0", len(d)))
d = d[d["area"] > 0]
buoc.append(("area > 0", len(d)))
d = d.drop_duplicates(subset=KHOA_TRUNG)
buoc.append(("Bỏ trùng theo KHOA_TRUNG", len(d)))

sau_ghep = (d.merge(tn25, on="tinh_cu", how="left", validate="many_to_one")
             .merge(bang_tra[["tinh_cu", "tinh_moi"]], on="tinh_cu", how="left", validate="many_to_one"))
assert len(sau_ghep) == len(d), "Ghép làm thay đổi số dòng"
buoc.append(("Sau khi ghép thu nhập + bảng tra", len(sau_ghep)))

phieu = pd.DataFrame(buoc, columns=["buoc", "so_dong_con_lai"])
phieu["so_dong_bi_loai"] = (-phieu["so_dong_con_lai"].diff()).fillna(0).astype(int)
display(phieu)
print("Đạt ≥ 20.000 dòng:", len(sau_ghep) >= 20_000)

theo_tinh_moi = sau_ghep["tinh_moi"].value_counts().rename("so_tin").to_frame()
theo_tinh_moi["it_tin"] = theo_tinh_moi["so_tin"] < NGUONG_IT_TIN
print("Số tỉnh mới có tin:", len(theo_tinh_moi), "| dưới ngưỡng", NGUONG_IT_TIN, "tin:",
      int(theo_tinh_moi["it_tin"].sum()))
print("Tỉnh mới không có tin căn hộ nào:", sorted(set(bang_tra["tinh_moi"]) - set(theo_tinh_moi.index)))
print("Tỷ lệ tin của TP.HCM + Hà Nội:",
      f"{theo_tinh_moi.loc[['TP. Hồ Chí Minh', 'Hà Nội'], 'so_tin'].sum() / len(sau_ghep) * 100:.1f}%")
theo_tinh_moi

,buoc,so_dong_con_lai,so_dong_bi_loai
0,Toàn bộ tin rao,3500744,0
1,Chỉ căn hộ,762800,2737944
2,price > 0,677895,84905
3,area > 0,677895,0
4,Bỏ trùng theo KHOA_TRUNG,326226,351669
5,Sau khi ghép thu nhập + bảng tra,326226,0


Đạt ≥ 20.000 dòng: True
Số tỉnh mới có tin: 32 | dưới ngưỡng 30 tin: 8
Tỉnh mới không có tin căn hộ nào: ['Cao Bằng', 'Lai Châu']
Tỷ lệ tin của TP.HCM + Hà Nội: 84.3%


,so_tin,it_tin
tinh_moi,,
TP. Hồ Chí Minh,137558,False
Hà Nội,137383,False
Đà Nẵng,12981,False
Hưng Yên,11422,False
Khánh Hòa,6049,False
Hải Phòng,4772,False
Quảng Ninh,3039,False
Tây Ninh,1987,False
Ninh Bình,1912,False


**Kết luận:** sau bộ lọc nháp và hai phép ghép còn **326.226 dòng**, gấp hơn 16 lần ngưỡng 20.000
→ **ĐẠT**. Kể cả nếu khoá trùng được làm chặt hơn (giữ lại nhiều tin hơn), số dòng chỉ tăng thêm.

Phễu: 3.500.744 tin → 762.800 căn hộ → 677.895 có giá > 0 → 677.895 có diện tích > 0 → 326.226 sau bỏ
trùng → 326.226 sau ghép (ghép không làm mất dòng).

**Giới hạn cần ghi vào bản đăng ký:**
- Chỉ **32/34** tỉnh mới có tin căn hộ; 2 tỉnh mới (Cao Bằng, Lai Châu) không có tin căn hộ nào.
- **8 tỉnh mới dưới 30 tin** (Hà Tĩnh 8, Vĩnh Long 6, và 6 tỉnh chỉ có 1 tin) → trung vị không đáng tin,
  phải đánh dấu và không xếp hạng.
- Dữ liệu dồn về hai đầu: TP.HCM 137.558 và Hà Nội 137.383 tin, chiếm khoảng 84% số dòng.

## Phần 9 - Tổng kết theo thứ tự điều kiện của đề

| # | Điều kiện (mục 3.11) | Phần trong notebook | Bằng chứng (con số từ notebook) | Trạng thái |
|---|---|---|---|---|
| 1 | Ít nhất hai nguồn độc lập được hợp nhất bằng mã | Phần 7 | Tin rao (TinixAI) + thu nhập (Cục Thống kê), cùng bảng tra; 63/63 khoá khớp sau `slug()`; 3.500.744 dòng trước = sau ghép, 0 dòng không khớp | ĐẠT |
| 2 | Có tệp ≥ 5 MB hoặc REST API có phân trang | Phần 1 | 10/10 tệp Parquet ≥ 5 MB (162,7-171,5 MB mỗi tệp, tổng 1.669,2 MB) | ĐẠT |
| 3 | Từ 20.000 dòng trở lên sau hợp nhất | Phần 8 | 326.226 dòng căn hộ sau lọc nháp và ghép | ĐẠT |
| 4 | Ít nhất 3 trong 4 khuyết tật | Phần 6 | Đủ 4/4: thiếu (`price` trống 218.494 + 57.320 ô bằng 0), trùng (113 trùng toàn cột; tới 1.551.238 theo khoá), sai kiểu (`price`, `published_at` là chuỗi), chưa chuẩn (31 cách ghi hướng, 7 dòng tổng hợp, tên tỉnh lệch) | ĐẠT |
| 5 | Có cột ngày hoặc giờ, và giấy phép cho phép học tập | Phần 4 + Phần 5 | `published_at` 01/06/2025 → 30/03/2026, ép kiểu không lỗi ô nào; CC BY-NC 4.0, "Bản quyền: Không", MIT | ĐẠT |

**Ghi chú mang sang bản đăng ký, thiết kế lớp và khâu làm sạch:**
- Sửa bản đăng ký: thu nhập là **JSON-stat**, giá trị đã là **số thực** (không phải chuỗi "7.713,43");
  `price` của tin rao mới là cột **chuỗi** cần ép kiểu (thẻ dữ liệu ghi sai là `float64`).
- `NguonPxWeb` phải bỏ ký tự NUL thừa ở cuối tệp trước khi `json.loads`.
- `slug()` → lớp `ChuanHoaDiaDanh` (chỗ dùng `@staticmethod` thật).
- `price == 0` coi là thiếu; ngưỡng diện tích, giá/m² và số tin tối thiểu mỗi tỉnh để trong `cau_hinh.yaml`.
- Khoá trùng cần làm chặt hơn trước khi dùng thật.
- Không bỏ bớt phần Parquet: mỗi phần là một khoảng thời gian.
- Giới hạn: 2/34 tỉnh mới không có tin căn hộ, 8 tỉnh dưới 30 tin.